# Your first fine-tune

In this tutorial you train Qwen/Qwen3.5-9B with LoRA on a few question-and-answer pairs, save the trained adapter, and generate answers from that exact checkpoint. It uses the SDK alone, one call at a time.

You need a Looper API key and a project, as in [Hello Looper](101_hello_looper.ipynb): put your key in the settings cell's `API_KEY`.

## Install the SDK and tokenizer

`transformers` provides tokenization only; this notebook never loads model weights locally. After upgrading a package that was already imported, restart the kernel and run the notebook again.

In [1]:
%pip install -q "goaly-looper==0.1.0b4" "transformers==5.19.0"

## Settings

The model, its tokenizer, where Looper is, your project and your API key, then the size of the run: the LoRA's rank, the number of optimizer steps and the longest sequence it trains.

In [2]:
MODEL = "Qwen/Qwen3.5-9B"
TOKENIZER = "Qwen/Qwen3.5-9B"
BASE_URL = "https://api.looper.goaly.ai"
PROJECT = "example"
API_KEY = "YOUR_API_KEY"
LORA_RANK = 32
STEPS = 10
MAX_SEQ_LEN = 512

## Connect to Looper

The config says where the service is and which project the session opens in. Nothing is sent until it is built.

In [3]:
import asyncio
from uuid import uuid4

from transformers import AutoTokenizer

from looper_sdk import (
    CrossEntropy,
    Datum,
    Lora,
    SamplerConfig,
    SamplingParams,
    ServiceClient,
    StateCheckpoint,
    TrainingConfig,
)

service_config = ServiceClient.Config(
    base_url=BASE_URL, project=PROJECT, api_key=API_KEY, display_name="first-sft"
)
print(f"Model: {MODEL}; LoRA rank: {LORA_RANK}")

Model: Qwen/Qwen3.5-9B; LoRA rank: 32


## Tokenize the training data

The SDK sends token ids, not text. Each sequence holds a question prompt, its answer, and the tokenizer's end token. A `Datum` pairs the tokens with one weight per token: 0 on the prompt, which is only context, and 1 on the answer and the end token, so only those contribute to the loss. The service predicts each token from the ones before it, so pass the whole sequence and shift nothing yourself.

Sampling will use the same `Question: ... / Answer:` format.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)

PAIRS = [
    ("What is the capital of France?", "Paris."),
    ("What is two plus three?", "Five."),
    ("What color is a clear daytime sky?", "Blue."),
    ("How many legs does a spider have?", "Eight."),
    ("What is the opposite of hot?", "Cold."),
    ("Which planet do we live on?", "Earth."),
    ("What do bees make?", "Honey."),
    ("How many days are in a week?", "Seven."),
]


def prompt_tokens(question):
    return tokenizer.encode(f"Question: {question}\nAnswer:", add_special_tokens=False)


data = []
for question, answer in PAIRS:
    prompt = prompt_tokens(question)
    reply = tokenizer.encode(f" {answer}", add_special_tokens=False)
    reply.append(tokenizer.eos_token_id)
    data.append(Datum(tokens=(*prompt, *reply), weights=(0,) * len(prompt) + (1,) * len(reply)))

batch_tokens = sum(len(datum.tokens) for datum in data)
print(f"{len(data)} examples, {batch_tokens} tokens per batch")

8 examples, 120 tokens per batch


## Train and save the LoRA adapter

Each step accumulates gradients with `forward_backward`, then applies them with `optim_step`. The printed loss is the step's loss sum divided by its weighted target count. The loss on this one repeated batch should fall; it does not measure generalization.

Starting the run can take several minutes, while GPUs are allocated and the model loads. `save_state` succeeds once the checkpoint is on the run's disk. `wait_checkpoint` returns once it is uploaded, which a sampler needs. You can skip that wait when you only go on training.

The whole run sits inside `async with`, so the session closes, and its GPUs are released, also when a call fails.

In [5]:
async def train_and_save() -> StateCheckpoint:
    training_config = TrainingConfig(
        model=MODEL,
        kind=Lora(rank=LORA_RANK),
        tokens_per_step=max(1024, MAX_SEQ_LEN, batch_tokens),
        max_seq_len=MAX_SEQ_LEN,
    )
    async with await service_config.build() as service:
        print("Starting the training run; GPU allocation and model loading can take minutes...")
        training_client = await service.create_training_client(training_config, timeout_s=20 * 60)
        print("Trainer is ready!")

        for _ in range(STEPS):
            # Accumulate this batch's gradients.
            await training_client.forward_backward(data, CrossEntropy())

            # Apply them, and wait for the step's result.
            update = await training_client.optim_step()
            step = await update.result(timeout_s=300)

            loss = step.loss_sum / step.weighted_targets
            print(f"Step {step.step:2d}: loss={loss:.4f}, grad_norm={step.grad_norm:.3f}")

        # A name belongs to the project, so a random one avoids conflicts.
        name = f"first-sft-{uuid4().hex[:8]}"
        saved = await (await training_client.save_state(name)).result(timeout_s=300)
        ready = await service.wait_checkpoint(saved.id, timeout_s=15 * 60)
        print(f"checkpoint {name!r}: {ready.state.value}, {ready.size_bytes} bytes")

    return saved


# Clear an earlier result, so a failed rerun cannot silently sample an older checkpoint.
checkpoint = None
checkpoint = await train_and_save()
print("Training session closed. Keep this checkpoint name:", checkpoint.name)

Starting the training run; GPU allocation and model loading can take minutes...
Trainer is ready!
Step  1: loss=3.1632, grad_norm=12.504
Step  2: loss=1.3143, grad_norm=9.061
Step  3: loss=0.8945, grad_norm=6.799
Step  4: loss=0.0185, grad_norm=0.233
Step  5: loss=0.0048, grad_norm=0.074
Step  6: loss=0.0040, grad_norm=0.087
Step  7: loss=0.0088, grad_norm=0.331
Step  8: loss=0.0046, grad_norm=0.154
Step  9: loss=0.0032, grad_norm=0.114
Step 10: loss=0.0022, grad_norm=0.087
checkpoint 'first-sft-3aa901d7': ready_to_use, 1137259483 bytes
Training session closed. Keep this checkpoint name: first-sft-3aa901d7


## Sample from the trained checkpoint

A new session starts a sampler of `checkpoint.name`, the adapter saved above. The service loads the base model with those trained LoRA factors. Starting it can take several minutes.

The questions come from the training set, to show the API. Every request is submitted before any result is awaited, so the sampler generates them together. `finally` stops the sampler, and leaving `async with` closes its session, also when a request fails.

In [6]:
QUESTIONS = [question for question, _ in PAIRS[:4]]


async def sample_checkpoint(name: str):
    params = SamplingParams(
        max_tokens=64,
        temperature=0.7,
        stop_token_ids=(tokenizer.eos_token_id,),
    )
    sampler_config = SamplerConfig(
        checkpoint_name=name,
        max_seq_len=MAX_SEQ_LEN,
        max_concurrent_sequences=8,
    )
    prompts = [prompt_tokens(question) for question in QUESTIONS]

    async with await service_config.build() as service:
        print(f"Starting a sampler of trained checkpoint {name!r}...")
        sampler = await service.create_sampling_client(sampler_config, timeout_s=20 * 60)
        try:
            # Submit every request, then wait for all the results.
            requests = await asyncio.gather(
                *(sampler.sample(prompt, params=params) for prompt in prompts)
            )
            results = await asyncio.gather(*(op.result(timeout_s=300) for op in requests))

            for question, result in zip(QUESTIONS, results, strict=True):
                print(f"\n{question}")
                for sequence in result.sequences:
                    answer = tokenizer.decode(sequence.token_ids, skip_special_tokens=True)
                    print(f"  {answer.strip()!r} (finish: {sequence.finish_reason.value})")
            return results
        finally:
            await sampler.stop()


sample_results = await sample_checkpoint(checkpoint.name)

Starting a sampler of trained checkpoint 'first-sft-3aa901d7'...

What is the capital of France?
  'Paris.' (finish: stop)

What is two plus three?
  'Five.' (finish: stop)

What color is a clear daytime sky?
  'Blue.' (finish: stop)

How many legs does a spider have?
  'Eight.' (finish: stop)


## Next steps

The adapter stays in your project under its name until you delete it.

- [Operations](103_operations.ipynb) shows how to submit steps ahead, so the GPUs never wait for your program.
- [Chat data](201_chat_data.ipynb) turns conversations into datums under the model's own chat template.
- [The SFT loop](301_sft_loop.ipynb) trains a file of conversations with the cookbook's loop.